# 🚀 Модуль 2. Урок 1. Реляционные базы данных и фильтрация данных (SELECT, WHERE, NULL)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_02_sql/lesson_01_db_intro_select_filtering.ipynb)

## 📌 Бизнес-контекст и методология
SQL — фундаментальный язык аналитика данных. В реляционной СУБД данные нормализованы по таблицам.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

db_path = ensure_dataset("ecommerce.db")
conn = sqlite3.connect(db_path)
print(f"🟢 Подключение к базе данных успешно установлено: {db_path}")


🟢 Подключение к базе данных успешно установлено: course\datasets\ecommerce.db


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(ensure_dataset("ecommerce.db"))

query = """
SELECT 
    customer_id,
    first_name,
    city,
    acquisition_channel,
    registration_date
FROM customers
WHERE city IN ('Moscow', 'Saint Petersburg')
  AND registration_date >= '2025-01-01'
ORDER BY registration_date DESC
LIMIT 5;
"""

df_res = pd.read_sql_query(query, conn)
df_res


,customer_id,first_name,city,acquisition_channel,registration_date
0,679,User_679,Moscow,Email Newsletter,2026-05-15
1,66,User_66,Moscow,Paid Ads,2026-05-12
2,1127,User_1127,Moscow,Paid Ads,2026-05-08
3,708,User_708,Saint Petersburg,Email Newsletter,2026-05-05
4,829,User_829,Moscow,Paid Ads,2026-05-05


### 🎯 Практическое задание 1: Поиск завершенных заказов через Apple/Mir Pay
Найдите все завершенные заказы со статусом `Completed` и методом оплаты `Apple Pay / Mir Pay`.

In [3]:
### ВАШ КОД ЗДЕСЬ
query_pay = """
SELECT 
    order_id,
    customer_id,
    order_date,
    status,
    payment_method
FROM orders
WHERE status = 'Completed'
  AND payment_method LIKE '%Apple Pay%'
LIMIT 10;
"""

df_pay = pd.read_sql_query(query_pay, conn)
print(df_pay.head(5))

assert not df_pay.empty, "Заказы должны быть найдены"
assert (df_pay['status'] == 'Completed').all(), "Статус должен быть Completed"
print("✅ Тест успешно пройден: фильтрация отработала безошибочно!")


   order_id  customer_id           order_date     status       payment_method
0     10001            1  2026-01-28 00:00:00  Completed  Apple Pay / Mir Pay
1     10008            4  2025-09-03 00:00:00  Completed  Apple Pay / Mir Pay
2     10018            9  2025-09-10 00:00:00  Completed  Apple Pay / Mir Pay
3     10021           10  2026-05-16 00:00:00  Completed  Apple Pay / Mir Pay
4     10027           13  2025-09-27 00:00:00  Completed  Apple Pay / Mir Pay
✅ Тест успешно пройден: фильтрация отработала безошибочно!


### 💡 Трехзначная логика SQL
Сравнение `col = NULL` всегда возвращает `UNKNOWN`. Используйте `col IS NULL` или `col IS NOT NULL`.